# 复现回测结果

验证本地回测与 `--from-predictions` 结果一致。

In [1]:
import os, sys, pickle, json
import pandas as pd
import numpy as np
sys.path.insert(0, '../code/src')
from backtest import ETFBacktester, run_backtest, run_backtest_from_predictions
import warnings
warnings.filterwarnings('ignore')

## 配置

In [2]:
MODEL_FILE = "best_model_sliding.pth"
DATA_PATH = "../etf_data/etf_74.csv"
CACHE_DIR = "../output/predictions_cache"
BT_CACHE_DIR = "../output/backtest_cache"
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(BT_CACHE_DIR, exist_ok=True)

START_DATE = "2026-04-01"
END_DATE = "2026-05-12"
TOP_K = 3
REBALANCE_DAYS = 5
POSITION_PCT = 0.95
INITIAL_CAPITAL = 100000

In [4]:
import glob
from tqdm import tqdm

BASE_DIR = "../model"
MODEL_TYPES = [
                "bayes_itransformer_74_3_2026-01-01_2026-03-31",
                "bayes_dlinear_74_3_2026-01-01_2026-03-31",
                "bayes_lstm_74_3_2026-01-01_2026-03-31",
                "bayes_gru_74_3_2026-01-01_2026-03-31",
                "bayes_patchtst_74_3_2026-01-01_2026-03-31",
                "bayes_mamba_74_3_2026-01-01_2026-03-31",
                "bayes_nlinear_74_3_2026-01-01_2026-03-31",
               ]
MODEL_DIR = f"../model/{MODEL_TYPES[0]}/exp_0"


EXPERIMENTS = []
for mt in MODEL_TYPES:
    cnt = 0
    for exp_dir in sorted(glob.glob(f"{BASE_DIR}/{mt}/exp_*")):
        if os.path.exists(f"{exp_dir}/best_model_sliding.pth"):
            EXPERIMENTS.append((exp_dir, "best_model_sliding.pth"))
        if os.path.exists(f"{exp_dir}/best_model.pth"):
            EXPERIMENTS.append((exp_dir, "best_model.pth"))
        if os.path.exists(f"{exp_dir}/best_model_ndcg.pth"):
            EXPERIMENTS.append((exp_dir, "best_model_ndcg.pth"))
        cnt += 1
    if cnt:
        print(f"{mt}: 共 {cnt} 个实验")
print(f"共 {len(EXPERIMENTS)} 个实验")

bayes_itransformer_74_3_2026-01-01_2026-03-31: 共 13 个实验
bayes_patchtst_74_3_2026-01-01_2026-03-31: 共 117 个实验
bayes_mamba_74_3_2026-01-01_2026-03-31: 共 7 个实验
bayes_nlinear_74_3_2026-01-01_2026-03-31: 共 64 个实验
共 600 个实验


## 遍历所有实验，一次性缓存 + 回测

In [5]:
import time

cached_data, cached_features = ETFBacktester.load_data_once(
    data_path=DATA_PATH,
    scaler_path=f'{MODEL_DIR}/scaler.pkl',
    feature_num='39',
    verbose=True,
)


加载并缓存数据: ../etf_data/etf_74.csv


特征工程: 100%|██████████| 74/74 [00:03<00:00, 21.94it/s]


数据缓存完成: 77987 条记录, 74 只股票


In [6]:


all_results = []
for exp_dir, mf in tqdm(EXPERIMENTS, desc="回测"):
    cache_key = f"{exp_dir}/{mf}"
    safe_name = cache_key.replace("\\", "/").replace('../', '').replace('./', '').replace('/', '_')
    cache_path = os.path.join(CACHE_DIR, f"{safe_name}.pkl")
    
    if not os.path.exists(cache_path):
        try:
            bt = ETFBacktester.from_cached_data(
                model_dir=exp_dir, cached_data=cached_data,
                cached_features=cached_features, device='cpu',
                model_file=mf, verbose=False,
            )
            preds = bt.generate_predictions_dict(start_date=START_DATE, end_date=END_DATE, rebalance_days=REBALANCE_DAYS, first_rebalance_date=START_DATE)
            with open(cache_path, 'wb') as f:
                pickle.dump(preds, f)
            del bt.model, bt
        except Exception as e:
            print(f'FAIL gen {cache_key}: {e}')
            continue
    else:
        with open(cache_path, 'rb') as f:
            preds = pickle.load(f)
    
    for mode in ['close', 'open']:
        bt_cache_key = f"{safe_name}_{mode}.pkl"
        bt_cache_path = os.path.join(BT_CACHE_DIR, bt_cache_key)
        if os.path.exists(bt_cache_path):
            with open(bt_cache_path, 'rb') as f:
                row = pickle.load(f)
            all_results.append(row)
            continue
        try:
            r = run_backtest_from_predictions(
                predictions_dict=preds, data_path=DATA_PATH,
                start_date=START_DATE, end_date=END_DATE,
                top_k=TOP_K, rebalance_days=REBALANCE_DAYS,
                position_pct=POSITION_PCT, initial_capital=INITIAL_CAPITAL,
                trade_mode=mode, verbose=False, log=False,
            )
            row = {
                'experiment': exp_dir.replace("\\", "/").split('/')[-2] + '/' + exp_dir.replace("\\", "/").split('/')[-1],
                'model_file': mf, 'trade_mode': mode,
                'return': r.strategy_return,
                'dd': r.max_drawdown,
                'hs300': r.hs300_return,
                'excess': r.excess_return,
                'win_rate': r.rebalance_stats.get('win_rate', 0),
                'avg_return': r.rebalance_stats.get('avg_return', 0),
                'rebalances': r.rebalance_stats.get('total', 0),
            }
            with open(bt_cache_path, 'wb') as f:
                pickle.dump(row, f)
            all_results.append(row)
        except Exception as e:
            print(f'FAIL backtest {cache_key} {mode}: {e}')

df = pd.DataFrame(all_results)
for mode in ['close', 'open']:
    sub = df[df['trade_mode'] == mode].sort_values('return', ascending=False).head(5)
    print(f'\n=== {mode} Top 5 ===')
    for _, r in sub.iterrows():
        print(f'  {r["experiment"]:35s} {r["model_file"]:25s} return={r["return"]:6.2f}%  dd={r["dd"]:5.2f}%  win={r["win_rate"]:5.1f}%  avg={r["avg_return"]:+5.2f}%')

回测:   0%|          | 0/600 [00:00<?, ?it/s]

回测: 100%|██████████| 600/600 [04:29<00:00,  2.23it/s]


=== close Top 5 ===
  bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_0 best_model_sliding.pth    return= 22.07%  dd= 3.94%  win= 75.0%  avg=+3.52%
  bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_0 best_model.pth            return= 22.07%  dd= 3.94%  win= 75.0%  avg=+3.52%
  bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_0 best_model_ndcg.pth       return= 22.07%  dd= 3.94%  win= 75.0%  avg=+3.52%
  bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_8 best_model_sliding.pth    return= 22.07%  dd= 3.94%  win= 75.0%  avg=+3.52%
  bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_8 best_model.pth            return= 22.07%  dd= 3.94%  win= 75.0%  avg=+3.52%

=== open Top 5 ===
  bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_49 best_model_sliding.pth    return= 19.97%  dd= 1.44%  win=100.0%  avg=+2.86%
  bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_88 best_model_sliding.pth    return= 19.95%  dd= 1.62%  win=100.0%  avg=+4.15%
  bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_27 best_model.pth          

In [7]:
df = pd.DataFrame(all_results)
for mode in ['close', 'open']:
    sub = df[df['trade_mode'] == mode].sort_values('avg_return', ascending=False).head(10)
    print(f'\n=== {mode} Top 10 by Avg Return ===')
    display(sub[['experiment', 'model_file', 'return', 'avg_return','dd', 'hs300', 'excess']])


=== close Top 10 by Avg Return ===


,experiment,model_file,return,avg_return,dd,hs300,excess
642,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_8,best_model_sliding.pth,22.07,3.52,3.94,9.53,12.55
82,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_0,best_model_ndcg.pth,22.07,3.52,3.94,9.53,12.55
80,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_0,best_model.pth,22.07,3.52,3.94,9.53,12.55
78,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_0,best_model_sliding.pth,22.07,3.52,3.94,9.53,12.55
644,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_8,best_model.pth,22.07,3.52,3.94,9.53,12.55
646,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_8,best_model_ndcg.pth,22.07,3.52,3.94,9.53,12.55
592,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_71,best_model_ndcg.pth,19.22,2.86,2.82,9.53,9.69
100,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_100,best_model_ndcg.pth,19.22,2.86,2.82,9.53,9.69
300,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_27,best_model_sliding.pth,19.22,2.86,2.82,9.53,9.69
302,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_27,best_model.pth,19.22,2.86,2.82,9.53,9.69



=== open Top 10 by Avg Return ===


,experiment,model_file,return,avg_return,dd,hs300,excess
697,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_88,best_model_sliding.pth,19.95,4.15,1.62,9.53,10.42
101,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_100,best_model_ndcg.pth,19.95,4.15,1.62,9.53,10.42
99,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_100,best_model.pth,19.95,4.15,1.62,9.53,10.42
97,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_100,best_model_sliding.pth,19.95,4.15,1.62,9.53,10.42
305,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_27,best_model_ndcg.pth,19.95,4.15,1.62,9.53,10.42
303,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_27,best_model.pth,19.95,4.15,1.62,9.53,10.42
325,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_30,best_model_sliding.pth,17.22,3.31,1.64,9.53,7.70
517,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_6,best_model_sliding.pth,16.68,3.14,1.67,9.53,7.15
519,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_6,best_model.pth,16.68,3.14,1.67,9.53,7.15
521,bayes_patchtst_74_3_2026-01-01_2026-03-31/exp_6,best_model_ndcg.pth,16.68,3.14,1.67,9.53,7.15
